In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = pd.read_csv("../data/clean/final_df_clean.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/clean/final_df_clean.csv'

In [ ]:
# separate enrolled students
df_enrolled = df[df["target"] == "Enrolled"].copy()

df_model = df[df["target"] != "Enrolled"].copy()

df_model["target"].value_counts()

In [ ]:
RANDOM_STATE = 42

X = df_model.drop(columns="target")
y = df_model["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

In [ ]:
#copied from the consolidated notebook
nominal_features = [
    "application_mode_group",
    "relationship_group",
    "course_name",
    "nationality_group",
    "mothers_occupation",
    "fathers_occupation"
]

ordinal_features = [
    "mothers_qualification",
    "fathers_qualification",
    "prev_qual_level"
]

binary_features = [
    "displaced",
    "debtor",
    "tuition_fees_up_to_date",
    "gender",
    "scholarship_holder",
    "international"
]

numerical_features = [
    "admission_grade",
    "age_at_enrollment",
    "curricular_units_1st_sem_credited",
    "curricular_units_1st_sem_enrolled",
    "curricular_units_1st_sem_evaluations",
    "curricular_units_1st_sem_approved",
    "curricular_units_1st_sem_grade",
    "curricular_units_1st_sem_without_evaluations",
    "curricular_units_2nd_sem_credited",
    "curricular_units_2nd_sem_enrolled",
    "curricular_units_2nd_sem_evaluations",
    "curricular_units_2nd_sem_approved",
    "curricular_units_2nd_sem_grade",
    "curricular_units_2nd_sem_without_evaluations",
    "unemployment_rate",
    "inflation_rate",
    "gdp"
]

In [ ]:
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaling", MinMaxScaler())
])

nominal_pipeline = Pipeline([
    ("encoding", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

numerical_pipeline = Pipeline([
    ("scaling", MinMaxScaler())
])

In [ ]:
preprocessor = ColumnTransformer([
    ("ordinal", ordinal_pipeline, ordinal_features),
    ("nominal", nominal_pipeline, nominal_features),
    ("numerical", numerical_pipeline, numerical_features),
    ("binary", "passthrough", binary_features)
])

In [ ]:
logistic_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

logistic_pipeline

## Logistic Regression - Model Training

In [ ]:
logistic_pipeline.fit(X_train, y_train)

## Model Evaluation - Training Data

In [ ]:
y_train_pred = logistic_pipeline.predict(X_train)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# calculate training metrics
print("Training accuracy:",
      round(accuracy_score(y_train, y_train_pred), 3))

print("Precision (Dropout):",
      round(precision_score(y_train, y_train_pred, pos_label="Dropout"), 3))

print("Recall (Dropout):",
      round(recall_score(y_train, y_train_pred, pos_label="Dropout"), 3))

print("F1 (Dropout):",
      round(f1_score(y_train, y_train_pred, pos_label="Dropout"), 3))

In [ ]:
print(classification_report(y_train, y_train_pred))

In [ ]:
# confusion matrix

cm = confusion_matrix(
    y_train,
    y_train_pred,
    labels=["Dropout", "Graduate"]
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["Dropout", "Graduate"],
    yticklabels=["Dropout", "Graduate"]
)

plt.title("Logistic Regression - Training Data")
plt.xlabel("Predicted Student Status")
plt.ylabel("Actual Student Status")
plt.show()

### Initial Training Results - Observations

- The Logistic Regression model achieved a training
  accuracy of 91.9%.
- Dropout precision is 94.5%.
- Dropout recall is 84.2%.
- The F1-score for dropout is 89.0%.
- The model identifies most dropout students in
  the training dataset.
- Cross-validation is needed to evaluate how well
  the model performs on unseen data.
- The first model uses the consolidated features
  without the newly created approval rate.